# Featureの削除

---

OpenHPC環境からFeature（計算ノードグループ）を削除します。実行中ジョブへの影響を確認したうえで、計算ノードの削除、構成定義の削除、Slurmへの反映を行います。

## 概要

このNotebookでは、既存のOpenHPC環境に対して以下の操作を行います：

1. 削除するFeatureの選択と影響確認（どのPartitionに影響するか）
2. 実行中ジョブの確認とノードのDrain
3. 計算ノードの削除（VCユニットの停止・削除）
4. mdx VMの削除（mdx環境の場合のみ）
5. 構成定義の削除（`slurm_features` / `slurm_partitions`）
6. Slurmへの反映（slurm.conf / hosts.ohpc）

### 削除パターン

**パターン1: Partitionに複数のFeatureが含まれる場合**
- Feature と Nodeset のみ削除
- Partition は維持（他の Feature は影響なし）
- 例: `gpu-partition` が `[ns-gpu-a100-aws, ns-gpu-a100-azure]` を持つ場合に `gpu-a100-azure` を削除

**パターン2: PartitionにFeatureが1つだけの場合**
- Feature を削除 → Partition が空になるため、Partition も自動的に削除
- 例: `highmem` Partition が `[ns-highmem-aws]` のみを持つ場合

### 前提条件

- 初期構築（010, 020）が完了していること
- 削除対象の Feature が存在すること（083 で追加または 010 で定義）
- VCCアクセストークンが有効であること

## 準備

既存のOpenHPC環境にアクセスするための準備を行います。

### VCCアクセストークンの入力

VCノードを操作するためのアクセストークンを入力してください。

In [ ]:
from getpass import getpass
vcc_access_token = getpass()

入力されたアクセストークンが正しいことを、実際にVCCにアクセスして確認します。

In [ ]:
from common import logsetting
from vcpsdk.vcpsdk import VcpSDK
vcp = VcpSDK(vcc_access_token)

### UnitGroup名の指定

構築環境の UnitGroup名を指定します。

> 「010-パラメータ設定.ipynb」と同じUnitGroup名(`ugroup_name`)を指定することで、既に入力したパラメータを引き継ぐことができます。

VCノードを作成時に指定した値を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

#### チェック

指定されたUnitGroup名が適切であることを確認します。

group_vars にfeature, partition 設定が記録されていることを確認します。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)

# group_vars にslurm_features, slurm_partitions が定義されていることを確認する
if not {"slurm_features", "slurm_partitions"} <= set(gvars.keys()):
    raise RuntimeError("Feature, Partition設定が未実施です")

Ansibleで接続できることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m ping

## 削除するFeatureの選択

### 対象Featureの指定

削除対象となるFeature名を指定します。

現在の環境に登録されているFeatureを確認します。補助情報として、それぞれのFeatureのプロバイダ名、ホスト名、所属するPartitionをあわせて表示します。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)
for name, cfg in gvars.get("slurm_features", {}).items():
    tp = cfg["hostname_template"]
    max_nodes = len(cfg["ip_addresses"])
    partitions = find_affected_partitions(gvars, name)
    print(f"""  - {name}
        Provider: {cfg['provider']}
        Hosts: {tp.format(1)} .. {tp.format(max_nodes)}
        Partitions: {', '.join(partitions)}
""")

上に表示されたFeature一覧から、削除するFeature名を指定してください。

In [ ]:
# (例)
# delete_feature_name = 'gpu-a100-azure'

delete_feature_name = 

### 削除後のPartition構成の確認

指定した Feature を削除した場合に、関連する Partition が維持されるか、Partition ごと削除されるかを確認します。Feature削除後もPartitionに他のNodesetが残る場合はPartitionが維持されます。

> default Partition が削除される場合はエラーになります。
> 先に「085-Partitionの変更.ipynb」で default を別の Partition に変更してください。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
affected_partitions, deleted_partitions = check_feature_deletion_impact(gvars, delete_feature_name)
feature_config = gvars["slurm_features"][delete_feature_name]
delete_vc_unit = feature_config.get("vc_unit", delete_feature_name)

print(f"""
削除対象 Feature: {delete_feature_name}
削除する VCユニット: {delete_vc_unit}
""")
for pname in affected_partitions:
    if pname in deleted_partitions:
        print(f"{pname}: Partition ごと削除されます")
    else:
        print(f"{pname}: Nodeset を削除")

### スケジュール設定の確認

削除対象のFeatureにノード数変更のスケジュール設定が行われている場合、Featureを削除できません。
スケジュール設定を先に削除する必要があります（「821-ノード数のスケジュール設定を削除する.ipynb」を使用）。

In [ ]:
_schedule_target = gvars.get("vcnode_schedule_target_feature")
if _schedule_target is not None and _schedule_target == delete_feature_name:
    raise RuntimeError(
        f"Feature '{delete_feature_name}' にはスケジュール設定が行われています。"
        "スケジュール対象のFeatureは削除できません。"
        "先に821でスケジュール設定を削除してください。"
    )

## 削除前の安全措置

VCノードを削除する前に、対象ノードで実行中のジョブがないかを確認します。

### 実行中ジョブの確認

現在投入されているジョブの状態を確認します。

In [ ]:
partitions_str = ','.join(affected_partitions)
if affected_partitions:
    !ansible {ugroup_name}_master -m shell \
        -a "squeue -p {partitions_str} \
        --format='%.18i %.9P %.8j %.8u %.8T %.10M %.9l %R' \
        | grep -v COMPLETED || true"

### ノードをDRAINに変更

削除対象ノードへの新規ジョブ投入を停止します。Slurm に登録されているノードを DRAIN 状態に設定します。

In [ ]:
feature_config = gvars["slurm_features"][delete_feature_name]
node_list = build_slurm_hostlist(
    feature_config["hostname_template"],
    len(feature_config["ip_addresses"]),
)
reason = f"removing nodeset ns-{delete_feature_name}"
!ansible {ugroup_name}_master -b -m shell \
    -a "scontrol update NodeName={node_list} State=DRAIN \
    Reason='{reason}' 2>&1 || true"

Slurmの状態を確認します。

In [ ]:
!ansible {ugroup_name}_master \
    -a "sinfo -p {partitions_str}"

### 対象ノードの残存ジョブ確認（オプション）

削除対象 Feature が属する Partition に投入されているジョブ数を確認する場合は、以下のセルを実行してください。表示された数が 0 であれば安全に削除できます。0 より大きい場合はこのセルを繰り返し実行し、ジョブが完了したことを確認してから次の手順に進んでください。

In [ ]:
if affected_partitions:
    !ansible {ugroup_name}_master -m shell \
        -a "squeue -h -p {partitions_str} | wc -l"

### Slurmのノード登録を解除

実際にVCノードを削除する前にSlurmのノード登録を解除しておきます。

In [ ]:
!ansible {ugroup_name}_master -b -m shell \
    -a "scontrol delete NodeName={node_list} 2>&1 || true"

Slurmの状態を確認します。

In [ ]:
!ansible {ugroup_name}_master \
    -a "sinfo -p {partitions_str}"

## 計算ノードの削除

削除対象のFeatureに対応するVCユニットを削除します。

削除前に、現在のVCユニットの一覧を確認しておきます。

In [ ]:
ug = vcp.get_ugroup(ugroup_name)
ug.df_units()

VCユニットを削除します。**この操作は元に戻せません**。

In [ ]:
import contextlib
from time import sleep

target_unit = ug.get_unit(delete_vc_unit)
for addr in target_unit.find_ip_addresses():
    with contextlib.suppress(Exception):
        target_unit.delete_nodes(ip_address=addr)
        sleep(5)

ug.delete_units(delete_vc_unit, force=True)

削除後のVCユニット一覧を確認します。

In [ ]:
ug.df_units()

ansibleのインベントリファイルから削除したノードのエントリを削除します。

In [ ]:
%run scripts/group.py
remove_group_from_inventory_yml(f"{ugroup_name}_{delete_vc_unit}", recursive=True)

更新後のインベントリの状態を確認します。

In [ ]:
!ansible-inventory --graph {ugroup_name}

## mdx VMの削除

mdx VMを削除します。mdx以外のクラウドを利用している場合はこの章をスキップしてください。

### mdx操作の準備

In [ ]:
from getpass import getpass
mdx_token = getpass("mdx API token")

IPv4接続を強制するように設定します。

In [ ]:
%run scripts/mdx_ops.py
use_ipv4_only()

mdxプラグインを読み込みます。

In [ ]:
from vcpsdk.plugins.mdx_ext import MdxResourceExt
mdx = MdxResourceExt(mdx_token)
mdx.set_current_project_by_name(gvars['mdx_project_name'])

### VM削除対象の特定

In [ ]:
# 削除するFeatureの全ノードのホスト名を算出
feature_config = gvars["slurm_features"][delete_feature_name]
hostname_template = feature_config["hostname_template"]
max_nodes = len(feature_config["ip_addresses"])
hosts_to_remove = [
    hostname_template.format(i + 1)
    for i in range(max_nodes)
]
print(f"削除対象VM: {hosts_to_remove}")

### VMの強制停止と削除

In [ ]:
from vcpsdk.plugins.mdx_ext import SLEEP_COUNT
from vcpsdk.plugins.mdx_ext import SLEEP_TIME_SEC
from time import sleep

# VM強制停止
for vm in hosts_to_remove:
    try:
        mdx.power_off_vm(vm, wait_for=False)
    except Exception as e:
        print(f"⚠ {vm} の停止でエラー: {e}")

for vm in hosts_to_remove:
    for i in range(SLEEP_COUNT):
        info = mdx.get_vm_info(vm)
        if info is None or info['status'] == 'PowerOFF':
            break
        sleep(SLEEP_TIME_SEC)
    else:
        raise RuntimeError(f'Error: VM {vm} not powered off.')

In [ ]:
# VM削除
for vm in hosts_to_remove:
    try:
        mdx.destroy_vm(vm, wait_for=False)
    except Exception as e:
        print(f"⚠ {vm} の削除でエラー: {e}")

for vm in hosts_to_remove:
    for i in range(SLEEP_COUNT):
        info = mdx.get_vm_info(vm)
        if info is None:
            print(f"✓ {vm} を削除しました")
            break
        sleep(SLEEP_TIME_SEC)
    else:
        raise RuntimeError(f'Error: VM {vm} not destroyed.')

## 構成定義の削除

`group_vars` から対象のFeature定義を削除します。また関連するPartition定義の内容も更新します。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)

# Partition の更新
affected_partitions = find_affected_partitions(gvars, delete_feature_name)
for pname in affected_partitions:
    remove_feature_from_partition(ugroup_name, pname, delete_feature_name)

# Feature の削除
delete_slurm_feature(ugroup_name, delete_feature_name)

`group_vars` ファイルの内容を確認します。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Slurmへの反映

構成定義の変更をSlurmの設定ファイルに反映し、その再読み込みを行います。

ansible で `slurm.conf` と `hosts.ohpc` を再生成し、Slurm に設定を再読み込みさせます。まずチェックモードで実行します。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v -CD playbooks/setup-slurm.yml

実際に変更を行います。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v playbooks/setup-slurm.yml

削除されたNodesetがslurm.confから除去されていることを確認します。

In [ ]:
# Nodeset が削除されていることを確認
!ansible {ugroup_name}_master -m shell \
    -a "grep -w 'NodeSet=ns-{delete_feature_name}' /etc/slurm/slurm.conf && echo 'NG: Nodeset が残っています' || echo 'OK: Nodeset が削除されました'"

# MaxNodeCount が更新されていることを確認
print("\n=== MaxNodeCount ===")
!ansible {ugroup_name}_master -m shell \
    -a "grep 'MaxNodeCount' /etc/slurm/slurm.conf"

# Partition 一覧の確認
print("\n=== Partition / Nodeset 一覧 ===")
!ansible {ugroup_name}_master -m shell \
    -a "grep -wE 'PartitionName|NodeSet' /etc/slurm/slurm.conf"

Slurmクラスタのノードの状態を確認します。

In [ ]:
!ansible {ugroup_name}_master \
    -a "sinfo -o '%P %N %F %f'"